In [1]:
import numpy as np

# Rating, some unknown

In [23]:
R0 = np.full([8,11], fill_value=np.nan)
R0[:4,:]= np.array([[4.5,4.5,4.5,2,3.5,3,5,2.5,4.5,5,4],
                    [4,4,4.5,1.5,4,3.5,5,2,4.5,5,4],
                    [5,5,3.5,4,2.5,2,4,5,4.5,4,3.5],
                    [4,5,4,2.5,3,3,4.5,3,5,4.5,3.5]])
R0[4,1]=5;R0[4,9]=3;
R0[5,2]=5;R0[5,6]=4.5;
R0[6,3]=2;R0[6,10]=4.5;
R0[7,4]=3.5; R0[7,6]=2;
R0

array([[4.5, 4.5, 4.5, 2. , 3.5, 3. , 5. , 2.5, 4.5, 5. , 4. ],
       [4. , 4. , 4.5, 1.5, 4. , 3.5, 5. , 2. , 4.5, 5. , 4. ],
       [5. , 5. , 3.5, 4. , 2.5, 2. , 4. , 5. , 4.5, 4. , 3.5],
       [4. , 5. , 4. , 2.5, 3. , 3. , 4.5, 3. , 5. , 4.5, 3.5],
       [nan, 5. , nan, nan, nan, nan, nan, nan, nan, 3. , nan],
       [nan, nan, 5. , nan, nan, nan, 4.5, nan, nan, nan, nan],
       [nan, nan, nan, 2. , nan, nan, nan, nan, nan, nan, 4.5],
       [nan, nan, nan, nan, 3.5, nan, 2. , nan, nan, nan, nan]])

In [24]:
# known entry=1; unknown entry=0
mask = np.ones(R0.shape);
mask[np.isnan(R0)]=0;
mask

array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0.]])

## fill with averages

In [27]:
np.set_printoptions(precision = 2, suppress = True)

In [28]:
col_avg = np.nanmean(R0, axis=0)
R0_fill = R0.copy()
for j in range(R0.shape[1]):
    R0_fill[np.isnan(R0[:,j]),j] = col_avg[j]
print(R0_fill)

[[4.5  4.5  4.5  2.   3.5  3.   5.   2.5  4.5  5.   4.  ]
 [4.   4.   4.5  1.5  4.   3.5  5.   2.   4.5  5.   4.  ]
 [5.   5.   3.5  4.   2.5  2.   4.   5.   4.5  4.   3.5 ]
 [4.   5.   4.   2.5  3.   3.   4.5  3.   5.   4.5  3.5 ]
 [4.38 5.   4.3  2.4  3.3  2.88 4.17 3.12 4.62 3.   3.9 ]
 [4.38 4.7  5.   2.4  3.3  2.88 4.5  3.12 4.62 4.3  3.9 ]
 [4.38 4.7  4.3  2.   3.3  2.88 4.17 3.12 4.62 4.3  4.5 ]
 [4.38 4.7  4.3  2.4  3.5  2.88 2.   3.12 4.62 4.3  3.9 ]]


## use SVD to fill again: with centering

In [70]:
R0_fill_c = R0_fill - np.mean(R0_fill, axis=0)
U,S,Vt = np.linalg.svd(R0_fill_c)
V = Vt.T
k = 4
Sm = np.diag(S[:k])
R1 = U[:,:k]@Sm@(V[:,:k]).T+col_avg
error = np.linalg.norm(R1*mask-R0_fill*mask)
print(R1)
print(error)

[[4.33 4.37 4.51 2.   3.53 3.07 5.01 2.65 4.53 4.98 4.04]
 [4.04 4.25 4.68 1.46 3.82 3.42 4.99 1.95 4.61 5.11 4.08]
 [5.   5.11 3.61 3.98 2.42 1.97 3.99 4.99 4.56 4.05 3.53]
 [4.04 4.89 3.92 2.54 3.1  3.02 4.5  2.97 4.92 4.44 3.42]
 [4.31 5.07 4.41 2.38 3.22 2.87 4.16 3.18 4.71 3.05 3.96]
 [4.47 4.62 4.48 2.27 3.38 2.87 4.52 3.08 4.52 4.22 4.08]
 [4.47 4.59 4.53 2.2  3.45 2.89 4.16 3.03 4.5  4.24 4.14]
 [4.35 4.71 4.26 2.37 3.48 2.88 2.   3.15 4.65 4.31 3.96]]
0.8831040457093396


## use SVD to fill again: without centering

In [71]:
U,S,Vt = np.linalg.svd(R0_fill)
V = Vt.T
k = 4
Sm = np.diag(S[:k])
R2=U[:,:k]@Sm@(V[:,:k]).T
error = np.linalg.norm(R2*mask-R0_fill*mask)
print(R2)
print(error)

[[4.27 4.45 4.46 2.03 3.53 3.14 5.   2.61 4.64 4.99 3.96]
 [3.98 4.17 4.6  1.43 3.77 3.37 4.98 1.91 4.53 5.08 4.01]
 [4.92 5.12 3.52 3.99 2.38 2.   3.99 4.95 4.59 4.04 3.44]
 [4.36 4.62 4.19 2.46 3.19 2.8  4.54 3.16 4.56 4.42 3.8 ]
 [4.3  4.99 4.41 2.35 3.18 2.81 4.16 3.17 4.62 3.03 3.96]
 [4.43 4.82 4.48 2.34 3.42 3.01 4.51 3.07 4.74 4.27 4.03]
 [4.37 4.74 4.46 2.25 3.44 3.   4.15 2.97 4.69 4.27 4.02]
 [4.37 4.68 4.3  2.36 3.47 2.85 2.   3.16 4.61 4.31 3.99]]
1.248329697205837
